# DTCC年度预告来源原件核验
只登记取得原件并视觉核对的2026年通知，区分Closed、Open1有限服务、Open2无结算。未来日程仅为已公布计划，不是实际服务观测。

In [1]:
from pathlib import Path
import os,json,hashlib,base64
import pandas as pd,qlib
from qlib.workflow import R
from pypdf import PdfReader
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/dtc_calendar_source_20261015';p=json.loads((card/'plan.json').read_text());source=root/p['source_manifest'];assert hashlib.sha256(source.read_bytes()).hexdigest()==p['source_manifest_sha256'];m=json.loads(source.read_text());raw=source.parent;pdf=(raw/m['path']).read_bytes();assert hashlib.sha256(pdf).hexdigest()==m['sha256'];assert base64.b32encode(hashlib.sha1(pdf).digest()).decode().rstrip('=')==m['capture']['digest']
reader=PdfReader(raw/m['path']);assert len(reader.pages)==1;text=reader.pages[0].extract_text();assert '23036-25' in text and 'limited settlement services' in text and 'will not provide settlement services' in text
# Transcribed from the visually inspected table; statuses preserve its footnotes.
entries=[('2026-01-01','New Year','Closed'),('2026-01-19','MLK','Closed'),('2026-02-16','Presidents','Closed'),('2026-04-03','Good Friday','Open1'),('2026-05-25','Memorial','Closed'),('2026-06-19','Juneteenth','Closed'),('2026-07-03','Independence observed','Open1'),('2026-09-07','Labor','Closed'),('2026-10-12','Columbus','Open2'),('2026-11-11','Veterans','Open2'),('2026-11-26','Thanksgiving','Closed'),('2026-12-25','Christmas','Closed')]
table=pd.DataFrame(entries,columns=['scheduled_date','holiday','notice_status']);table['meaning']=table.notice_status.map({'Closed':'closed','Open1':'limited_services_specific_notice_required','Open2':'open_without_settlement'});table['past_at_retrieval']=pd.to_datetime(table.scheduled_date).le(pd.Timestamp(m['as_of']));table['actual_service_verified']=False
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/pdf','inputs/archive_indices','inputs/extracted_text']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_manifest=m,pdf=pdf,archive_indices=json.loads((raw/'archive_indices.json').read_text()),extracted_text=text);rec.save_objects(artifact_path='analysis',anticipated_holidays=table);rec.log_metrics(verified_pdf_pages=1,scheduled_holiday_rows=12,registered_artifact_count=6)
summary={'experiment':card.name,'decision':'needs_data_remaining_years_and_specific_service_notices','new_fits':0,'new_backtests':0,'verified_pdf_notices':1,'notice_year':2026,'scheduled_holiday_rows':12,'complete_calendar_qualified':False,'broker_value_dates_qualified':False,'settlement_calendar_provider_built':False,'provider_uri':str(provider),'provider_role':'existing Qlib context only; no new market or settlement calendar claimed','region':'us','as_of':m['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
table.to_csv(card/'anticipated_holidays.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/'research/ibkr_interest_archive_20261015/nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=m['sha256'],raw_authority='Archived official DTCC notice 23036-25; CDX payload digest matched');(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table.to_string(index=False))


# DTCC年度结算预告原件：2026已取得，其余年份仍缺

本轮取得DTCC通知23036-25的单页PDF原件，文件签名为PDF，文内签发日2025-10-15，主题为2026年预期假日日程。原始网址为https://www.dtcc.com/-/media/Files/pdf/2025/10/15/23036-25.pdf ，通过2026-04-01 18:21:51 UTC的Internet Archive捕获取得；Memento时间一致，下载内容SHA1/Base32与CDX摘要ALQ5VTCACE7XVZL6A57X7AV55PLULNRL一致，另保存SHA256。捕获时间不是首次发布时间认证。

按PDF技能渲染并视觉检查了全部表格和脚注，转录12行预告日期，未把HTML错误页当作PDF。正文明确每个假日前还会另发具体服务通知，因此年度预告本身不提供所有服务的最终实际状态。

关键区分：Closed为关闭；Open1为有限结算服务、需核查具体服务；Open2为开放但不提供结算。2026年Good Friday（4月3日）和独立日观察日（7月3日）标Open1，不能直接视为普通股票可结算。Columbus Day（10月12日）和Veterans Day（11月11日）标Open2，也不能把设施开放当作可结算。其他列出的8个日期为Closed。完整转录见anticipated_holidays.csv；超过2026-10-14的日期只作为已公布计划保留，不是假称未来实际观测。

本轮对2023–2026四个通知的Globals路径和media PDF路径分别查询存档，共8个路径族；7个无匹配捕获，2026 media路径获得上述原件。未再次请求已返回404的实时原件端点。空索引结果不证明其他路径或其他服务无历史资料；2023–2025本次路径没有被认证。

决策needs_data_remaining_years_and_specific_service_notices。没有把年度表扩展成完整自然日/交易日日历，没有认证券商实际本金及费用value date，也没有把原件中的有限服务日直接写成true。现有Qlib provider仅作Recorder运行上下文，本轮未创建或宣称新的合格结算日历provider。

0拟合、0新增回测、未给原收益计息、生产未改。下一项仍需要其余年份原件及服务级例外通知，尤其要区分普通股票CNS与有限交割服务。来源原件、提取文本、搜索索引和转录表已进入Qlib Recorder与NAS，冷恢复逐对象哈希通过。数据资格补齐不等于稳定净SOXX超额，目标未完成。
